In [4]:
import itertools
import numpy as np
from photon_canon import Medium, System
from photon_canon.lut import generate_lut, LUT

In [5]:
# Make water medium
di_water = Medium(n=1.33, mu_s=0, mu_a=0, g=0, desc='di water')
glass = Medium(n=1.523, mu_s=0, mu_a=0, g=0, desc='glass')
# Start the system
system = System(di_water, 0.2,  # 1mm
                glass, 0.017,  # 0.17mm
                surrounding_n=1.33)
variable = Medium(n=1.33, mu_s=0, mu_a=0, g=1, desc='tissue')  # Placeholder to update at iteration
system.add(variable, 0.2)

# Init parameter sets
mu_s_array = np.arange(0, 101, 1)
mu_a_array = np.arange(1, 102, 1)
g_array = [0.9]
d = float('inf')
n = 10000
recurse = False
wl0 = 700

In [ ]:
# Generate a photon object (either directly or through the system illumination)
photon = system.beam(batch_size=n, recurse=recurse)
simulation_id = generate_lut(system, 
                             variable, 
                             {'mu_s': mu_s_array, 'mu_a': mu_a_array, 'g': g_array}, 
                             photon,
                             pbar=True)

mu_s: 0 - mu_a: 67 - g: 0.9:   1%|          | 66/10201 [01:00<2:31:53,  1.11it/s]

In [ ]:
lut = LUT(dimensions=['mu_s', 'mu_a'], simulation_id=simulation_id)

In [ ]:
from matplotlib import cm
import matplotlib.pyplot as plt
X, Y, Z = lut.surface()

fig = plt.figure(figsize=[10, 10])
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(Y, X, Z, cmap=cm.hot)
ax.plot_wireframe(Y, X, Z, rstride=10, cstride=10, color='k')
# ax.invert_yaxis()
ax.set_xlabel('mu_a')
ax.set_ylabel('mu_s')
ax.set_zlabel('Detected reflectance')
fig.tight_layout()
plt.show()

## Comparison to diffusion approximation

In [ ]:
from tqdm import tqdm

mu_s_array = np.arange(0, 50, 1)
mu_a_array = np.arange(1, 50, 1)
tissue_n = 1.33
collection_n = 1.33
for mu_s, mu_a in tqdm(itertools.product(mu_s_array, mu_a_array), 
                       desc='Diffusion approximation', total=len(mu_s_array) * len(mu_a_array)):
    R = diffusion_appoximation(mu_s=mu_s, mu_a=mu_a, n_tissue=tissue_n, n_collection=collection_n)